In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as f
import os

In [ ]:
os.listdir('/kaggle/input/')

In [ ]:
directory_txt = '/kaggle/input/datasets/ashyou09/contract-understanding-atticus-dataset-cuad/full_contract_txt 2/full_contract_txt'
combined_text = []
for filename in os.listdir(directory_txt)[:100]:
    if filename.endswith('.txt'):
        with open(os.path.join(directory_txt, filename), 'r', encoding='utf-8') as file:
            combined_text.append(file.read())
            
text = "\n\n[NEW_CONTRACT]\n\n".join(combined_text)
print(f"Dataset loaded. Total characters: {len(text)}")

In [ ]:
batch_size = 32
block_size = 256
max_iters = 5000
eval_interval = 500
eval_iters = 200
learning_rate = 3e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
d_model = 512
n_head = 8
n_layer = 8
dropout = 0.1

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
atoi = {ch:i for i, ch in enumerate(chars)}
itoa = {i:ch for i, ch in enumerate(chars)}

def encode(s):
    return [atoi[i] for i in s]

def decode(i):
    return ''.join([itoa[a] for a in i])

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.8 * len(data))
train_data = data[:n]
validation_data = data[n:]

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.num_heads = num_heads
        self.head_size = head_size
        self.qkv = nn.Linear(d_model, 3 * num_heads * head_size, bias=False)
        self.proj = nn.Linear(head_size * num_heads, d_model)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.num_heads, self.head_size).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_size).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_size).transpose(1, 2)
        scores = q @ k.transpose(-2, -1) * (self.head_size ** -0.5)
        scores = scores.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        scores = f.softmax(scores, dim=-1)
        scores = self.dropout(scores)
        out = scores @ v
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        out = self.dropout(self.proj(out))
        return out


class MultiLayerPerceptron(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Transformer(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        head_size = d_model // n_head
        self.attention = MultiHeadAttention(n_head, head_size)
        self.mlp = MultiLayerPerceptron(d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, d_model)
        self.position_embedding_table = nn.Embedding(block_size, d_model)
        self.blocks = nn.Sequential(*[Transformer(d_model, n_head=n_head) for i in range(n_layer)])
        self.final_norm = nn.LayerNorm(d_model) # final layer norm
        self.de_embd = nn.Linear(d_model, vocab_size)
        self.apply(self.weights)

    def weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if getattr(module, 'bias', None) is not None:
                torch.nn.init.zeros_(module.bias)
                
    def forward(self, idx, targets=None):
        B, T = idx.shape
        token_embedded = self.token_embedding_table(idx)
        position_embedded = self.position_embedding_table(torch.arange(T, device=device))
        x = token_embedded + position_embedded
        x = self.blocks(x)
        x = self.final_norm(x)
        logits = self.de_embd(x)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = f.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_tokens, temperature=1.0, top_k=5):
        for i in range(max_tokens):
            idx_cropped = idx[:, -block_size:]
            logits, loss = self(idx_cropped)
            logits = logits[:, -1, :]
            logits = logits / temperature
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = float('-inf')
            prob = f.softmax(logits, dim=-1)
            idx_next = torch.multinomial(prob, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [ ]:
model = GPT()
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model.to(device)
print(sum(p.numel() for p in model.parameters()))

In [ ]:
def get_batch(s_type):
    data = train_data if s_type == 'train' else validation_data
    pos = torch.randint(len(data) - block_size, (batch_size,))
    train = torch.stack([data[i:i + block_size] for i in pos])
    result = torch.stack([data[i + 1:i + block_size + 1] for i in pos])
    train, result = train.to(device), result.to(device)
    return train, result

@torch.no_grad()
def calculate_loss():
    model.eval()
    out = {}
    loss_train = torch.zeros(eval_iters)
    loss_val = torch.zeros(eval_iters)
    for i in range(eval_iters):
        X1, Y1 = get_batch('train')
        X2, Y2 = get_batch('val')
        pred1, loss1 = model(X1, Y1)
        pred2, loss2 = model(X2, Y2)
        if loss1.numel() > 1:
            loss1 = loss1.mean()
        if loss2.numel() > 1:
            loss2 = loss2.mean()
        loss_train[i], loss_val[i] = loss1.item(), loss2.item()
    out['train'] = loss_train.mean()
    out['val'] = loss_val.mean()
    model.train()
    return out

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for i in range(max_iters):
    if i % eval_interval == 0 or i == max_iters - 1:
        losses = calculate_loss()
        print(f"Iteration {i}: train loss {losses['train']:.2f}, validation loss {losses['val']:.2f}")
    x, y = get_batch('train')
    logits, loss = model(x, y)
    if loss.numel() > 1:
        loss = loss.mean()
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

In [ ]:
model.eval()
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.module.generate(context, max_tokens=500, temperature = 1.2)[0].tolist()))

In [ ]:
pre_context = "Section 1. Definitions."
context_tokens = [atoi[ch] for ch in pre_context]
input_tensor = torch.tensor(context_tokens, dtype=torch.long, device=device).unsqueeze(0)
model.eval()
with torch.no_grad():
    generated_tokens = model.module.generate(input_tensor, max_tokens=1000, temperature=0.7, top_k=3)[0].tolist()
print("--- GENERATED OUTPUT ---")
print(decode(generated_tokens))

In [ ]:
torch.save(model.state_dict(), 'legal_gpt_weights.pth')